In [1]:
import numpy as np
import time, os, sys
from urllib.parse import urlparse
#import skimage.io
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
from PIL import Image
from urllib.parse import urlparse
from cellpose import models, core
from cellpose import models
from cellpose import plot
from cellpose import utils
from cellpose import io, metrics
from matplotlib.pyplot import figure
use_GPU = core.use_gpu()
print('>>> GPU activated? %d'%use_GPU)
import tifffile
# call logger_setup to have output of cellpose written
from cellpose.io import logger_setup
from scipy.spatial import Voronoi, voronoi_plot_2d
from scipy import ndimage
import scipy.spatial
import scipy
import pandas as pd
logger_setup();
from tqdm import tqdm

>>> GPU activated? 1
2024-10-14 18:42:14,025 [INFO] WRITING LOG OUTPUT TO /home/hpc/iwi5/iwi5171h/.cellpose/run.log
2024-10-14 18:42:14,026 [INFO] 
cellpose version: 	3.0.9.dev8+gc03958c 
platform:       	linux 
python version: 	3.8.19 
torch version:  	2.3.1


In [2]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [3]:
def process_image(image_path, model, diameter):
    # Load image and ground truth coordinates
    image = tifffile.imread(image_path)
    h, w = image.shape

    # Evaluate the model to get the predicted mask
    mask, flow, style = model.eval(image, 
                                   channels=[0,0],
                                   diameter=diameter,
                                   flow_threshold=0,
                                   cellprob_threshold=0.0)
    
    

    # Calculate the center coordinates from the predicted mask
    result_cofocal = ndimage.center_of_mass(mask, mask, np.unique(mask))
    resultList_cofocal = list(result_cofocal) 
    df_cofocal = pd.DataFrame(resultList_cofocal, columns=['row', 'column'])
    
    # Calculate number of pixel in each mask (ie cell)
    unique_labels, counts = np.unique(mask, return_counts = True)
    pixel_counts = dict(zip(unique_labels, counts))
    df_cofocal['mask_label'] = df_cofocal.index
    df_cofocal.drop(0, inplace=True, axis=0)  # Remove background NA
    df_cofocal['pixel_count'] = df_cofocal['mask_label'].map(pixel_counts) # number of pixel in each cell
    
    # Calculate area in square micron for each mask
    microns_per_pixel = 0.5639772144372924  # microns per pixel from metadata
    #mm_per_pixel = microns_per_pixel / 1000
    df_cofocal['mean cell area'] = df_cofocal['pixel_count'] * (microns_per_pixel ** 2) # area of each cone cell "μm²"
    
    # rearrange columns
    df_result = df_cofocal[['mask_label', 'row', 'column', 'pixel_count', 'mean cell area']].reset_index(drop=True)
    
    return df_result, h, w

In [4]:
def process_all_images(base_folder, high_model, low_model, high_diameter, low_diameter):
    # Use glob to find and sort the image file
    image_paths = sorted(glob(os.path.join(base_folder, 'AOSLO', '*.tiff')))

    microns_per_pixel = 0.5639772144372924
    mm_per_pixel = microns_per_pixel / 1000
    
    # Initialize lists to store the lengths
    cell_density = []
    average_areas = []
    degree = []
    cell_count = []
  
    # Process each image and its corresponding ground truth coordinates
    for image_path in tqdm(image_paths):
        
        try:
            # Extract filename from the full path
            filename = os.path.basename(image_path)
            
            if 'calculated' in filename:
                # Split filename into parts using '_' # Extract the relevant parts
                parts = filename.split('_')
                value1 = int(parts[4])
                value2 = int(parts[5])

                # Filter based on degrees
                if -4 < value1 < 4 and -4 < value2 < 4: # for low degree only
                    model = low_model
                    diameter = low_diameter
                    
                if not (-4 < value1 < 4 and -4 < value2 < 4): # for high degree only
                    model = high_model
                    diameter = high_diameter
                       
                df_result, h, w = process_image(image_path, model, diameter)
                N = len(df_result) # number of cells
                area_mm2 = h * w * (mm_per_pixel ** 2)
                density = N / area_mm2
                
                average_area = np.sum(df_result['mean cell area'])  / N

                cell_density.append(density)
                average_areas.append(average_area)
                degree.append((value1,value2))
                cell_count.append(N)
        
        except (ValueError, IndexError):
            continue
        
    return cell_count, cell_density, average_areas, degree

In [5]:
# Create model object
high_degree_model_path = '/home/hpc/iwi5/iwi5171h/thesis_src/trained_models/cellpose_models/CP_cyto_high_degree_10_images'#high degree
low_degree_model_path = '/home/hpc/iwi5/iwi5171h/thesis_src/trained_models/cellpose_models/CP_cyto_low_degree_10_images'#low degree

high_diameter = 18 # Set the appropriate diameter
high_model = models.CellposeModel(gpu=True, pretrained_model=high_degree_model_path)

low_diameter = 7
low_model = models.CellposeModel(gpu=True, pretrained_model=low_degree_model_path)

2024-10-14 18:42:14,097 [INFO] ** TORCH CUDA version installed and working. **
2024-10-14 18:42:14,098 [INFO] >>>> using GPU
2024-10-14 18:42:14,290 [INFO] >>>> loading model /home/hpc/iwi5/iwi5171h/Thesis_src/trained_models/cellpose_models/CP_cyto_high_degree_10_images
2024-10-14 18:42:14,702 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2024-10-14 18:42:14,703 [INFO] >>>> model diam_labels =  12.205 (mean diameter of training ROIs)
2024-10-14 18:42:14,706 [INFO] ** TORCH CUDA version installed and working. **
2024-10-14 18:42:14,707 [INFO] >>>> using GPU
2024-10-14 18:42:14,815 [INFO] >>>> loading model /home/hpc/iwi5/iwi5171h/Thesis_src/trained_models/cellpose_models/CP_cyto_low_degree_10_images
2024-10-14 18:42:15,194 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)
2024-10-14 18:42:15,195 [INFO] >>>> model diam_labels =  8.268 (mean diameter of training ROIs)


In [6]:
AOSLO_list = [
    '4078_right_retina_Confocal_RPE_2023-09-09',
    '5181_left_retina_Confocal_RPE_IS_2023-09-10',
    '8323_right_retina_confocal_RPE_2023-09-13',
    '2538_right_retina_Confocal_RPE_2023-09-10',
    '4017_left_retina_Confocal_RPE_2023-09-10',
    '4064_od_retina_montaged_2023',
    '4571_left_retina_Confocal_RPE_IS_2023_09_09',
    '5160_right_retina_2024',
    '5160_right_retina_Confocal_RPE_2023-09-10',
    '5163_right_retina_RPE_2023-06-08',
    '5163_right_retina_2024',
    '5165_right_retina_RPE_2023-06-08',
    '5188_right_retina_Confocal_RPE_2023-09-10',
    '5225_right_retina_2024',
    '5232_right_retina_2024',
    '5213_right_retina_2024_session198',
    '5214_right_retina_2024_session205'
    
]

output_dir = 'analysis/csv_files'
if not os.path.exists(output_dir):
        os.makedirs(output_dir)

for AOSLO in tqdm(AOSLO_list):
    
    #base_folder = f'/home/hpc/iwi5/iwi5171h/data_processing/original_data/{AOSLO}/'
    base_folder = f'/home/hpc/iwi5/iwi5171h/thesis_src/multimodal_AOSLO_data/{AOSLO}/'
    
    
    # Process each AOSLO
    cell_count, pred_density, average_areas, degree = process_all_images(base_folder, high_model, low_model, high_diameter, low_diameter)
    
    # Creating a DataFrame 
    data = pd.DataFrame({
        'Degree': degree,
        'Cell Count': cell_count,
        'Predicted Cell Density': np.log10(pred_density),
        'Average Cell Area': average_areas
    })

    data = data.sort_values(by='Degree')
    data.reset_index(drop=True, inplace=True)
    
    # Save data to a CSV file
    data.to_csv(f'{output_dir}/{AOSLO}.csv', index=False)
    print(f'Processed and saved: {AOSLO}.csv')

  0%|          | 0/17 [00:11<?, ?it/s]


KeyboardInterrupt: 